<a href="https://colab.research.google.com/github/hiba-essid/Antenna-Di-signer/blob/main/docs/code/baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Partie 2 : votre baseline ResNet18 sur CIFAR-10

Ce notebook suit la **partie 2** de l'énoncé : <https://bonben.github.io/ensta-tp-deep-learning/02_baseline/>.

Sur Colab, vérifiez d'abord que vous êtes sur GPU : menu *Exécution > Modifier le type
d'exécution > GPU T4*.

Les cellules marquées **TODO** sont à écrire. Presque tout ce qu'il faut est dans le
tutoriel de la partie 1, [*Training a Classifier*](https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html) : chaque TODO vous dit dans
quelle section aller chercher. Gardez-le ouvert dans un autre onglet.

Juste après chaque TODO, une cellule **Vérification** vous dit si vous êtes sur la bonne
voie : exécutez-la, elle doit afficher `OK`.

## 0. Le code du réseau et les données (fourni)

In [1]:
# Télécharge resnet.py (la description du réseau) et CIFAR-10 (copie rapide)
# dans le dossier de travail, s'ils n'y sont pas déjà.
import os
import tarfile
import urllib.request

SITE = "https://bonben.github.io/ensta-tp-deep-learning"
CIFAR_URL = "https://github.com/bonben/ensta-tp-deep-learning/releases/download/cifar10/cifar-10-python.tar.gz"

if not os.path.exists("resnet.py"):
    urllib.request.urlretrieve(f"{SITE}/code/resnet.py", "resnet.py")
if not os.path.exists("data/cifar-10-batches-py"):
    urllib.request.urlretrieve(CIFAR_URL, "cifar.tar.gz")
    with tarfile.open("cifar.tar.gz") as tar:
        tar.extractall("data", filter="data")
    os.remove("cifar.tar.gz")
print("resnet.py et CIFAR-10 sont prêts")

resnet.py et CIFAR-10 sont prêts


## Imports et réglages (fourni)

In [2]:
import time
import torch
import torch.nn as nn
import torchvision
from torchvision.transforms import v2
import matplotlib.pyplot as plt
from resnet import ResNet18

# Les DataLoader lancent des processus auxiliaires (num_workers). Cette ligne garantit
# qu'ils fonctionnent aussi si le notebook est un jour exporté en script.
torch.multiprocessing.set_start_method("fork", force=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Calcul sur :", device)

# Moyenne et écart-type de CIFAR-10, canal par canal
MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2435, 0.2616)

BATCH_SIZE = 128
LR = 0.1
EPOCHS = 2          # 2 pour mettre au point ; 30 pour l'entraînement complet (étape 8)
DATA_DIR = "./data"

Calcul sur : cuda


## 1. Les données

In [3]:
# TODO 1 : les données.
# Modèle : la section « 1. Load and normalize CIFAR10 » du tutoriel.
# Différences avec le tutoriel :
#   - DEUX transformations au lieu d'une :
#       transform_train = v2.Compose([v2.RandomCrop(32, padding=4), v2.RandomHorizontalFlip(),
#                                     v2.ToImage(), v2.ToDtype(torch.float32, scale=True),
#                                     v2.Normalize(MEAN, STD)])
#       transform_val : la même, SANS RandomCrop ni RandomHorizontalFlip
#   - root=DATA_DIR et batch_size=BATCH_SIZE
#   - les noms : train_set, val_set, train_loader, val_loader
#     (le tutoriel les appelle trainset, testset, trainloader, testloader)

transform_train = v2.Compose([
    v2.RandomCrop(32, padding=4),
    v2.RandomHorizontalFlip(),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(MEAN, STD)
])

transform_val = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(MEAN, STD)
])

train_set = torchvision.datasets.CIFAR10(root=DATA_DIR, train=True,
                                        download=True, transform=transform_train)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE,
                                          shuffle=True, num_workers=2)

val_set = torchvision.datasets.CIFAR10(root=DATA_DIR, train=False,
                                      download=True, transform=transform_val)
val_loader = torch.utils.data.DataLoader(val_set, batch_size=BATCH_SIZE,
                                        shuffle=False, num_workers=2)

In [5]:
# Vérification 1
x, y = next(iter(train_loader))
print(len(train_set), len(val_set), x.shape, y.shape)
assert len(train_set) == 50000 and len(val_set) == 10000
assert x.shape == (BATCH_SIZE, 3, 32, 32) and y.shape == (BATCH_SIZE,)
print("OK")

50000 10000 torch.Size([128, 3, 32, 32]) torch.Size([128])
OK


## 2. Le modèle

In [4]:
# TODO 2 : le modèle.
#   - model : un ResNet18() envoyé sur le GPU
#     (comme « net = Net() » dans la section « 2. Define a Convolutional Neural Network »,
#      puis « net.to(device) » dans la section « Training on GPU »)
#   - n_params : son nombre total de paramètres (votre réponse à la question Q3 de la partie 1)

In [ ]:
# Vérification 2
print(f"{n_params:,} paramètres")
out = model(x.to(device))
print(out.shape)
assert n_params == 11_173_962 and out.shape == (BATCH_SIZE, 10)
print("OK")

## 3. Perte, optimiseur, scheduler

In [ ]:
# TODO 3 : remplacer les ... (section « 3. Define a Loss function and optimizer » du tutoriel)
criterion = ...   # la même perte que dans le tutoriel
optimizer = torch.optim.SGD(model.parameters(), lr=..., momentum=..., weight_decay=...)
#   valeurs à mettre : lr=LR, momentum=0.9, weight_decay=5e-4

# Fourni : le scheduler fait décroître le taux d'apprentissage de LR à 0 en EPOCHS époques
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
# Vérification 3
assert isinstance(criterion, nn.CrossEntropyLoss), "criterion doit être nn.CrossEntropyLoss()"
group = optimizer.param_groups[0]
assert (group["lr"], group["momentum"], group["weight_decay"]) == (LR, 0.9, 5e-4), group
print("OK")

## 4. Une époque d'entraînement

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    """Une passe sur tout le jeu d'entraînement. Renvoie (perte moyenne, précision en %)."""
    model.train()  # met le modèle en mode entraînement (utile pour la BatchNorm)
    total_loss, correct, total = 0.0, 0, 0
    for inputs, labels in loader:
        # TODO 4.a : envoyer inputs et labels sur le GPU
        #   (section « Training on GPU ») :
        #       inputs, labels = inputs.to(device), labels.to(device)

        # TODO 4.b : remise à zéro des gradients, passe avant, perte, rétropropagation, pas
        #   (section « 4. Train the network » : les lignes de optimizer.zero_grad()
        #    à optimizer.step(), avec model à la place de net)

        # TODO 4.c : compter les bonnes réponses du batch
        #   (section « 5. Test the network on the test data », dans le bloc qui affiche
        #    « Accuracy of the network on the 10000 test images ») :
        #       _, predicted = torch.max(outputs, 1)
        #       correct += (predicted == labels).sum().item()

        # Fourni : loss est la perte MOYENNE du batch ; on la multiplie par la taille
        # du batch pour faire une somme, qu'on divise par total à la fin.
        total_loss += loss.item() * labels.size(0)
        total += labels.size(0)
    return total_loss / total, 100 * correct / total

## 5. L'évaluation

In [ ]:
def evaluate(model, loader, criterion):
    """Évalue le modèle SANS l'entraîner. Renvoie (perte moyenne, précision en %)."""
    model.eval()  # met le modèle en mode évaluation (utile pour la BatchNorm)
    total_loss, correct, total = 0.0, 0, 0
    # TODO 5 : recopier ici le bloc du tutoriel qui affiche
    #   « Accuracy of the network on the 10000 test images »
    #   (section « 5. Test the network on the test data », juste après la phrase
    #    « Let us look at how the network performs on the whole dataset »).
    #   Recopiez de « with torch.no_grad(): » jusqu'à la ligne « correct += ... » incluse,
    #   avec model à la place de net et loader à la place de testloader, et ajoutez :
    #     - juste après « images, labels = data » :
    #         images, labels = images.to(device), labels.to(device)
    #     - juste après « outputs = model(images) », le calcul de la perte :
    #         total_loss += criterion(outputs, labels).item() * labels.size(0)

    return total_loss / total, 100 * correct / total

In [ ]:
# Vérification 5 : un réseau non entraîné répond au hasard
val_loss, val_acc = evaluate(model, val_loader, criterion)
print(f"perte {val_loss:.2f}, précision {val_acc:.1f} %")
assert 1.5 < val_loss < 4 and 3 < val_acc < 20, "attendu : perte proche de 2,3 et précision proche de 10 %"
print("OK")

## 6. La boucle d'entraînement

In [ ]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_acc = 0.0

for epoch in range(EPOCHS):
    t0 = time.time()

    # TODO 6.a : entraîner une époque, avec votre fonction train_one_epoch
    #   train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)

    # TODO 6.b : évaluer sur le jeu de validation, avec votre fonction evaluate
    #   val_loss, val_acc = evaluate(...)          (quels arguments ? regardez sa définition)

    # TODO 6.c : faire avancer le scheduler d'un cran
    #   scheduler.step()

    # TODO 6.d : ranger les quatre valeurs dans history
    #   history["train_loss"].append(train_loss)
    #   ... et de même pour train_acc, val_loss et val_acc

    # TODO 6.e : si val_acc est plus grand que best_acc :
    #   - mettre à jour best_acc
    #   - sauvegarder les poids (comme dans le tutoriel, « Let's quickly save our trained model ») :
    #       torch.save(model.state_dict(), "best.pt")

    # Fourni : affichage
    print(f"époque {epoch + 1}/{EPOCHS} | train {train_loss:.3f} / {train_acc:.1f} % | "
          f"val {val_loss:.3f} / {val_acc:.1f} % | {time.time() - t0:.0f} s")

## Les courbes (fourni)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(history["train_loss"], label="train")
ax1.plot(history["val_loss"], label="val")
ax1.set(xlabel="époque", ylabel="perte")
ax2.plot(history["train_acc"], label="train")
ax2.plot(history["val_acc"], label="val")
ax2.set(xlabel="époque", ylabel="précision (%)")
for ax in (ax1, ax2):
    ax.legend()
    ax.grid(alpha=0.3)
fig.savefig("curves.png", dpi=100)
plt.show()

## 7. Recharger le modèle sauvegardé

In [ ]:
# TODO 7 : recharger les poids de "best.pt" dans un NOUVEAU modèle, et l'évaluer.
#   Comme dans le tutoriel (section 5, « net = Net() » puis « net.load_state_dict(...) ») :
#       reloaded = ResNet18().to(device)
#       reloaded.load_state_dict(torch.load("best.pt", map_location=device))
#   puis évaluez-le sur val_loader avec votre fonction evaluate :
#       _, reloaded_acc = evaluate(...)

In [ ]:
# Vérification 7
print(f"meilleure précision pendant l'entraînement : {best_acc:.2f} %, modèle rechargé : {reloaded_acc:.2f} %")
assert abs(reloaded_acc - best_acc) < 0.01
print("OK")